# 半监督 GNN 空间转录组细胞类型注释

**流程**：scRNA（有标签）+ Xenium（无标签）联合训练，Domain Adaptation

**修复清单**：P0①②③ / P1④⑤⑥⑦ / P2⑧⑨⑩ / P3⑪⑫⑬

In [ ]:
# ============================================================
# Cell 1: 环境配置
# ============================================================

import os, sys, warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# 路径管理
PATHS = {
    "cache": {
        "flex":   "./data/cache/flex_data_processed.rds",
        "xenium": "./data/cache/xenium_data_processed.rds",
    },
    "output": {
        "models":      "./results/models/",
        "predictions": "./results/predictions/",
        "plots":       "./plots/",
    },
}
for d in PATHS["output"].values():
    os.makedirs(d, exist_ok=True)

# ──── 全局超参（所有可调参数集中管理）────────────────────────────────
PARAMS = {
    # 图构建
    "knn_k":          30,
    "val_ratio":      0.2,

    # 模型架构
    "hidden_dim":     256,
    "proj_dim":       512,      # P3-⑭：投影层（3000→512→256），缓解跨度
    "dropout":        0.5,

    # 训练
    "n_epochs":       500,      # 半监督需要更多 epoch
    "lr":             1e-3,
    "weight_decay":   5e-4,
    "patience":       40,       # P2：patience 放宽

    # LR 调度器（P2-⑨）
    "lr_factor":      0.5,
    "lr_patience":    15,
    "min_lr":         1e-5,

    # 梯度裁剪（P2-⑩）
    "max_grad_norm":  1.0,

    # 热身阶段（仅 CE + MMD，不加伪标签）
    "warmup_epochs":  30,

    # Domain Adaptation 权重（P1-⑤）
    "lambda_mmd":     0.1,
    "lambda_ent":     0.01,
    "lambda_pl":      0.3,

    # 伪标签阈值（P3-⑫）
    "pl_threshold":   0.90,
    "pl_update_freq": 10,

    # 硬件
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed":   42,
}

from utils import set_seed
set_seed(PARAMS["seed"])

print(f"Device: {PARAMS['device']}")
print(f"PyTorch: {torch.__version__}")
print("Params configured ✓")


## Step 1: 数据加载（R → Python）

从 `labelTransfer.ipynb` 生成的缓存中加载数据，取 Xenium panel 共同基因。

In [ ]:
# ============================================================
# Cell 2: 数据加载（%%R 魔法命令）
# ============================================================

%%R -o flex_expr -o xenium_expr -o flex_labels -o cell_types -o xenium_coords

library(Seurat)

cat("Loading cached data...\n")
flex_data   <- readRDS("./data/cache/flex_data_processed.rds")
xenium_data <- readRDS("./data/cache/xenium_data_processed.rds")

# ── 特征对齐：只使用 Xenium panel 基因（P0 核心要求）──────────────
# 错误做法：intersect(rownames(flex_data), rownames(xenium_data)) 后用 scRNA 全基因
# 正确做法：以 Xenium panel 为基准，scRNA 也只看这些基因
xenium_panel <- rownames(xenium_data)
common_genes <- intersect(rownames(flex_data), xenium_panel)
cat("Xenium panel:", length(xenium_panel), "| Common genes:", length(common_genes), "\n")

# ── 提取原始 counts（log_normalize 在 Python 做，保持一致性）─────
flex_counts   <- as.matrix(GetAssayData(flex_data,   layer="counts")[common_genes, ])
xenium_counts <- as.matrix(GetAssayData(xenium_data, layer="counts")[common_genes, ])

# 转置为 (n_cells, n_genes)
flex_expr   <- t(flex_counts)
xenium_expr <- t(xenium_counts)

# 细胞类型标签
flex_labels_raw <- flex_data$cell_type
unique_labels   <- unique(flex_labels_raw)
label_map       <- setNames(seq_along(unique_labels) - 1L, unique_labels)
flex_labels     <- as.integer(label_map[flex_labels_raw])
cell_types      <- unique_labels

# Xenium 空间坐标（μm）
xenium_coords <- as.matrix(data.frame(
    x = xenium_data@images[[1]]@boundaries$centroids@coords[, 1],
    y = xenium_data@images[[1]]@boundaries$centroids@coords[, 2]
))

cat("scRNA cells:", nrow(flex_expr), "| Xenium cells:", nrow(xenium_expr), "\n")
cat("Cell types:", length(cell_types), "\n")
cat(cell_types, sep="  ")


## Step 2: 数据预处理

修复 P0-① 和 P1-⑥：统一归一化 + 生物学标准化

In [ ]:
# ============================================================
# Cell 3: 预处理（log1p + 统一归一化）
# ============================================================

from utils import log_normalize, unified_normalize, build_combined_dataset

# ── P1-⑥ 生物学标准化（必须在 StandardScaler 之前）────────────────
print("Step 1: log1p normalization (library-size + log1p)...")
flex_log   = log_normalize(flex_expr)
xenium_log = log_normalize(xenium_expr)

print(f"  scRNA shape:  {flex_log.shape}")
print(f"  Xenium shape: {xenium_log.shape}")

# ── P0-① 统一归一化（只在 scRNA 上 fit）──────────────────────────
print("\nStep 2: unified StandardScaler (fit on scRNA only)...")
flex_norm, xenium_norm, fitted_scaler = unified_normalize(flex_log, xenium_log)

# ── P1-⑦⑧ 构建联合图（分层采样 + 类别权重）─────────────────────
print("\nStep 3: building combined dataset...")
data, class_weights, split_info = build_combined_dataset(
    flex_norm, xenium_norm, flex_labels,
    k=PARAMS["knn_k"],
    val_ratio=PARAMS["val_ratio"],
)

print(f"\nClass weights: {class_weights.numpy().round(3)}")
print("Preprocessing complete ✓")


## Step 3: 训练 GNN 模型（GCN / GraphSAGE / GAT）

In [ ]:
# ============================================================
# Cell 4: 训练三种 GNN 模型
# ============================================================

from models import GCN, GraphSAGE, GAT, run_experiment

GNN_CONFIGS = [
    (GCN,       "GCN"),
    (GraphSAGE, "GraphSAGE"),
    (GAT,       "GAT"),
]

gnn_results = {}
for model_class, model_name in GNN_CONFIGS:
    result = run_experiment(
        model_class, model_name,
        data, list(cell_types), PARAMS, class_weights,
    )
    gnn_results[model_name] = result

    # 保存模型权重
    import torch
    torch.save(
        result["model"].state_dict(),
        f"{PATHS['output']['models']}/{model_name}_best.pt"
    )

print("\n✅ All GNN models trained.")


## Step 4: TopACT Baseline（SVM + 空间平滑）

In [ ]:
# ============================================================
# Cell 5: TopACT
# ============================================================

from topact import TopACT

topact = TopACT(
    C=10.0,
    gamma="scale",
    kernel="rbf",
    spatial_weight=0.3,
    n_neighbors=10,
)

# 使用与 GNN 相同的 scaler（特征对齐）
print("Fitting TopACT SVM on scRNA data...")
topact.fit(flex_log, flex_labels, fitted_scaler=fitted_scaler)

# 推断 Xenium
print("Predicting Xenium cell types...")
topact_pred_idx, topact_proba = topact.predict(
    xenium_log,
    spatial_coords=xenium_coords,
    return_proba=True,
)
topact_confidence = topact_proba.max(axis=1)

topact_results = {
    "pred_indices": topact_pred_idx,
    "labels":       [cell_types[i] for i in topact_pred_idx],
    "probs":        topact_proba,
    "confidence":   topact_confidence,
}

print(f"TopACT prediction complete. "
      f"Median confidence: {topact_confidence.median():.3f}")


## Step 5: 评估（scRNA 验证集 — 唯一合法的 ground truth）

> ⚠️ **P0-④**：Seurat 标签只用于「事后对比分析」，不参与模型选择/调参。

In [ ]:
# ============================================================
# Cell 6: 评估（scRNA 验证集）
# ============================================================

import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report
from eval import compute_metrics

val_idx    = split_info["val_idx"]
val_true   = flex_labels[val_idx]

# ── 获取每个方法在验证集上的预测 ────────────────────────────────────
val_preds = {}

for name, res in gnn_results.items():
    model  = res["model"].to("cpu")
    data_c = data.to("cpu")
    model.eval()
    with torch.no_grad():
        log_probs = model(data_c)
    val_pred_idx = log_probs[data_c.val_mask].argmax(dim=1).numpy()
    val_preds[name] = val_pred_idx

# TopACT 验证集预测
topact_val_pred, _ = topact.predict(flex_log[val_idx], return_proba=True)
val_preds["TopACT"] = topact_val_pred

# ── 打印指标 ─────────────────────────────────────────────────────────
print("=" * 65)
print("  Method Comparison on scRNA Val Set (Real Ground Truth)")
print("=" * 65)
print(f"{'Method':<12} {'Acc':>7} {'F1-Mac':>8} {'F1-Wei':>8} {'Kappa':>8}")
print("-" * 65)

for method, y_pred in val_preds.items():
    m = compute_metrics(val_true, y_pred, n_classes=len(cell_types))
    print(f"{method:<12} {m['accuracy']:>7.4f} {m['f1_macro']:>8.4f} "
          f"{m['f1_weighted']:>8.4f} {m['kappa']:>8.4f}")
    if method == list(val_preds.keys())[-1]:
        print("=" * 65)

# ── 最佳模型详细报告 ─────────────────────────────────────────────────
best_method = max(val_preds,
                  key=lambda m: compute_metrics(val_true, val_preds[m])["f1_macro"]
                  if m in gnn_results else 0)
print(f"\nBest GNN: {best_method}")
print(classification_report(
    val_true, val_preds[best_method],
    target_names=list(cell_types), zero_division=0
))


## Step 6: Moran's I 空间自相关评估

In [ ]:
# ============================================================
# Cell 7: Moran's I（空间自相关）
# ============================================================

from topact import TopACT

morans_results = {}

all_xenium_preds = {name: res["pred_indices"] for name, res in gnn_results.items()}
all_xenium_preds["TopACT"] = topact_results["pred_indices"]

for method, pred_idx in all_xenium_preds.items():
    print(f"Computing Moran's I for {method}...", end=" ")
    global_mi = TopACT.morans_i(pred_idx, xenium_coords, n_neighbors=15)
    per_class  = TopACT.per_class_morans_i(
        pred_idx, xenium_coords, list(cell_types), n_neighbors=15
    )
    morans_results[method] = per_class
    print(f"Global I = {global_mi['I']:.4f}  (p={global_mi['p_value']:.4f})")

print("\nMoran's I computation complete ✓")


## Step 7: 生成所有论文图表（Fig 1–10）

In [ ]:
# ============================================================
# Cell 8: 生成全部论文图表
# ============================================================

from eval import generate_all_thesis_figures

# 为 Fig 2 (UMAP) 准备 scRNA 隐层嵌入
import torch

scrna_embeddings_per_model = {}
for name, res in gnn_results.items():
    model  = res["model"].to("cpu")
    data_c = data.to("cpu")
    model.eval()
    with torch.no_grad():
        h, _ = model.encode(data_c)
    # 前 n_flex 行是 scRNA，后面是 Xenium
    n_flex = data_c.n_flex
    scrna_h  = h[:n_flex].numpy()
    xenium_h = h[n_flex:].numpy()
    # 把两部分合并放入 embeddings（eval.py 会根据 n_flex 切分）
    gnn_results[name]["embeddings"] = np.vstack([scrna_h, xenium_h])

generate_all_thesis_figures(
    gnn_results        = gnn_results,
    topact_results     = topact_results,
    scrna_expr_norm    = flex_norm,
    scrna_labels       = flex_labels,
    xenium_coords      = xenium_coords,
    cell_types         = list(cell_types),
    val_labels         = val_true,
    val_pred_per_method= val_preds,
    morans_results     = morans_results,
    output_dir         = PATHS["output"]["plots"],
)


## Step 8: 保存最终预测结果

In [ ]:
# ============================================================
# Cell 9: 保存结果
# ============================================================

import pandas as pd, json

# ── 构建预测 DataFrame ──────────────────────────────────────────────
pred_df = pd.DataFrame(index=range(data.n_xenium))

for name, res in gnn_results.items():
    pred_df[f"{name}_label"]      = res["predictions"]
    pred_df[f"{name}_confidence"] = res["confidence"]
    proba_cols = pd.DataFrame(
        res["probabilities"],
        columns=[f"{name}_prob_{ct}" for ct in cell_types]
    )
    pred_df = pd.concat([pred_df, proba_cols], axis=1)

pred_df["TopACT_label"]      = topact_results["labels"]
pred_df["TopACT_confidence"] = topact_results["confidence"]

if xenium_coords is not None:
    pred_df["x_coord"] = xenium_coords[:, 0]
    pred_df["y_coord"] = xenium_coords[:, 1]

out_csv = f"{PATHS['output']['predictions']}/all_predictions.csv"
pred_df.to_csv(out_csv, index=False)
print(f"Predictions saved → {out_csv}")

# ── 实验摘要 ─────────────────────────────────────────────────────────
summary = {
    "cell_types":   list(cell_types),
    "n_scrna":      int(data.n_flex),
    "n_xenium":     int(data.n_xenium),
    "n_genes":      int(data.x.shape[1]),
    "params":       {k: str(v) for k, v in PARAMS.items()},
    "val_metrics":  {},
}

from eval import compute_metrics
for method, y_pred in val_preds.items():
    m = compute_metrics(val_true, y_pred, n_classes=len(cell_types))
    summary["val_metrics"][method] = {
        "accuracy":    round(float(m["accuracy"]),    4),
        "f1_macro":    round(float(m["f1_macro"]),    4),
        "f1_weighted": round(float(m["f1_weighted"]), 4),
        "kappa":       round(float(m["kappa"]),       4),
    }

summary_path = f"{PATHS['output']['predictions']}/experiment_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"Summary saved    → {summary_path}")

print("\n" + "="*60)
print("  实验完成！")
print("="*60)
print(f"  图表目录 : {PATHS['output']['plots']}")
print(f"  预测文件 : {out_csv}")
print(f"  摘要文件 : {summary_path}")


## (可选) 事后分析：与 Seurat 标签对比

> ⚠️ **仅用于定性对比，不参与调参或模型选择（P0-④）。**

In [ ]:
# ============================================================
# Cell 10: 事后 Seurat 对比（可选，不影响主实验）
# ============================================================

try:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    pandas2ri.activate()

    ro.r("""
        obj  <- readRDS('./results/seurat/xenium_annotated_final.rds')
        seurat_labels <- as.character(obj$predicted.id)
    """)
    seurat_labels_raw = list(ro.r("seurat_labels"))

    # 将字符串标签映射为整数
    ct_list  = list(cell_types)
    ct2idx   = {ct: i for i, ct in enumerate(ct_list)}
    seurat_idx = np.array([ct2idx.get(l, -1) for l in seurat_labels_raw])
    valid      = seurat_idx >= 0
    print(f"Seurat labels loaded: {valid.sum()} / {len(seurat_idx)} valid")

    print("\n─── Agreement with Seurat (qualitative only) ───")
    for method, pred_idx in {
        **{n: r["pred_indices"] for n, r in gnn_results.items()},
        "TopACT": topact_results["pred_indices"],
    }.items():
        agree = (pred_idx[valid] == seurat_idx[valid]).mean()
        print(f"  {method:<12}: agreement = {agree:.4f}")

    print("\n⚠️  上述数字仅供参考，不代表真实准确率（Seurat 本身也有噪声）。")

except Exception as e:
    print(f"Seurat comparison skipped ({e})")
